# Asignación de Hubs – Método 2 (KMedoids)
Asigna cada carga al hub más cercano del método 2 usando haversine.
- **Fuente de hubs**: `final_results.csv`
- **Fuente de cargas**: `parquetloadsfull.parquet`
- **Salida**: `hubs_assign_m2.parquet`

In [1]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

## 1. Cargar hubs método 2

In [2]:
hubs_m2 = pd.read_csv('../5. merge dbscan/results/final_results.csv')
print(hubs_m2.shape)
print(hubs_m2['categoría'].value_counts())
hubs_m2.head()

(192, 11)
categoría
terciario     104
primaria       57
secundaria     31
Name: count, dtype: int64


,id,lat,lng,Z,new_name,state,fam_id,categoría,cluster,total_loads_local,cluster_size
0,1.0,33.455575,-86.839677,1.000000,Pico AL,AL,familia_14,primaria,NaN,NaN,NaN
1,69.0,30.718925,-88.106887,0.583708,Pico AL,AL,familia_12,secundaria,NaN,NaN,NaN
2,80.0,32.399324,-86.311673,0.436066,Pico AL,AL,familia_18,secundaria,NaN,NaN,NaN
3,3.0,34.739338,-92.289498,1.000000,Pico AR,AR,familia_11,primaria,NaN,NaN,NaN
4,4.0,36.218294,-94.151811,0.894564,Pico AR,AR,familia_13,primaria,NaN,NaN,NaN


In [3]:
# Separar tiers
primaria   = hubs_m2[hubs_m2['Z'] == 1.0].copy()
secundaria = hubs_m2[(hubs_m2['Z'] < 1.0) & (hubs_m2['new_name'].notna())].copy()
terciario  = hubs_m2[hubs_m2['new_name'].isna()].copy()

print(f'Primaria: {len(primaria)}, Secundaria: {len(secundaria)}, Terciario: {len(terciario)}')

Primaria: 47, Secundaria: 41, Terciario: 104


In [4]:
# Construir tabla unificada de hubs con hub_id, hub_name, lat, lng, tier
def build_hub_table(primaria, secundaria, terciario):
    rows = []
    for _, r in primaria.iterrows():
        rows.append({'hub_id': str(r['id']), 'hub_name': r['new_name'],
                     'lat': r['lat'], 'lng': r['lng'],
                     'state': r.get('state', None), 'tier': 'primaria',
                     'cluster': r.get('cluster', None)})
    for _, r in secundaria.iterrows():
        rows.append({'hub_id': str(r['id']), 'hub_name': r['new_name'],
                     'lat': r['lat'], 'lng': r['lng'],
                     'state': r.get('state', None), 'tier': 'secundaria',
                     'cluster': r.get('cluster', None)})
    for _, r in terciario.iterrows():
        cid = str(int(r['cluster'])) if pd.notna(r.get('cluster')) else str(_)
        rows.append({'hub_id': f'C{cid}', 'hub_name': f'Cluster_{cid}',
                     'lat': r['lat'], 'lng': r['lng'],
                     'state': r.get('state', None), 'tier': 'terciario',
                     'cluster': r.get('cluster', None)})
    return pd.DataFrame(rows)

hubs_table = build_hub_table(primaria, secundaria, terciario)
hubs_table = hubs_table.dropna(subset=['lat', 'lng']).reset_index(drop=True)
print(f'Total hubs en tabla: {len(hubs_table)}')
hubs_table.head()

Total hubs en tabla: 192


,hub_id,hub_name,lat,lng,state,tier,cluster
0,1.0,Pico AL,33.455575,-86.839677,AL,primaria,NaN
1,3.0,Pico AR,34.739338,-92.289498,AR,primaria,NaN
2,10.0,Pico AZ,33.449091,-112.180692,AZ,primaria,NaN
3,11.0,Pico CA,33.973121,-118.133117,CA,primaria,NaN
4,16.0,Pico CO,39.788721,-104.881077,CO,primaria,NaN


## 2. Función de asignación haversine

In [5]:
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return 3958.8 * 2 * atan2(sqrt(a), sqrt(1 - a))

def assign_hub_m2(lat, lng, hubs_df, dist_list=[50, 100, 150]):
    """Retorna (near_hub_id, hub_name, hubs_50m, hubs_100m, hubs_150m)"""
    h = hubs_df.copy()
    h['dist'] = h.apply(lambda r: haversine(lat, lng, r['lat'], r['lng']), axis=1)
    nearest = h.loc[h['dist'].idxmin()]
    result = {'near_hub_m2': nearest['hub_id'], 'near_hub_name_m2': nearest['hub_name']}
    for d in dist_list:
        result[f'hubs_{d}m_m2'] = list(h[h['dist'] <= d]['hub_id'])
    return result


## 3. Cargar cargas

In [6]:
df = pd.read_parquet('../2. cleaning_loads/parquetloadsfull.parquet')
df = df.reset_index(drop=True)
n_original = len(df)
print(f'Total cargas: {n_original:,}')
df.head()

Total cargas: 57,573,489


,ID,Posted,PickUp,CityOrigin,LatOrigin,LngOrigin,CityDestination,LatDestination,LngDestination,Distance,Source,Equip,StateOrigin,HubOrigin,StateDestination,HubDestination,RatePerMile,Weight,Size,Broker_Shipper
0,0.0,2022-06-28,2022-07-01,Coquille,43.1802,-124.1841,Fremont,37.5265,-121.9852,530.0,DOFT,V,OR,Grants Pass (OR),CA,Stockton (CA),NaN,44000.0,NaN,NaN
1,1.0,2022-06-28,2022-06-30,Victorville,34.5277,-117.3536,Woodburn,45.1473,-122.8603,973.0,DOFT,V,CA,Los Angeles (CA),OR,Salem (OR),NaN,44000.0,NaN,NaN
2,2.0,2022-06-28,2022-06-28,Stockton,37.9765,-121.3109,Bend,44.0562,-121.3087,483.0,DOFT,V,CA,Stockton (CA),OR,Salem (OR),NaN,17000.0,NaN,NaN
3,3.0,2022-06-28,2022-07-01,Olivehurst,39.0817,-121.5549,Hermiston,45.8323,-119.2858,613.0,DOFT,V,CA,Stockton (CA),OR,Yakima (WA),NaN,27000.0,NaN,NaN
4,4.0,2022-06-28,2022-07-06,Moreno Valley,33.9244,-117.2045,Albany,44.6274,-123.0966,958.0,DOFT,V,CA,San Diego (CA),OR,Salem (OR),NaN,41000.0,NaN,NaN


## 4. Asignar hubs a orígenes

In [7]:
# Orígenes únicos con coordenadas
orig_uniq = (
    df[['LatOrigin', 'LngOrigin']]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f'Orígenes únicos con coords: {len(orig_uniq):,}')

Orígenes únicos con coords: 391,989


In [8]:
# Asignar (puede tardar varios minutos)
orig_results = orig_uniq.apply(
    lambda r: assign_hub_m2(r['LatOrigin'], r['LngOrigin'], hubs_table),
    axis=1,
    result_type='expand'
)
orig_uniq = pd.concat([orig_uniq, orig_results], axis=1)
orig_uniq.head()

,LatOrigin,LngOrigin,near_hub_m2,near_hub_name_m2,hubs_50m_m2,hubs_100m_m2,hubs_150m_m2
0,43.1802,-124.1841,C2,Cluster_2,[],[C2],[C2]
1,34.5277,-117.3536,11.0,Pico CA,[],"[11.0, C10]","[11.0, C10, C4]"
2,37.9765,-121.3109,C16,Cluster_16,[C16],"[86.0, C16]","[86.0, 64.0, C16, C18]"
3,39.0817,-121.5549,C16,Cluster_16,[],"[C16, C18]","[86.0, 64.0, C16, C18]"
4,33.9244,-117.2045,11.0,Pico CA,[],"[11.0, C10, C4]","[11.0, C14, C10, C4]"


In [9]:
# Verificar que orig_uniq no tiene duplicados en las keys (si hubiera, el merge aumentaría las filas)
assert orig_uniq.duplicated(subset=['LatOrigin', 'LngOrigin']).sum() == 0, \
    "orig_uniq tiene duplicados en (LatOrigin, LngOrigin)!"

df = df.merge(
    orig_uniq,
    on=['LatOrigin', 'LngOrigin'],
    how='left'
)

assert len(df) == n_original, f"Row count cambió tras merge orígenes: {len(df):,} != {n_original:,}"
print(f'✓ Filas después del merge orígenes: {len(df):,} (igual a original)')
print(f'  Orígenes con hub m2: {df["near_hub_m2"].notna().sum():,} / {n_original:,}')

✓ Filas después del merge orígenes: 57,573,489 (igual a original)
  Orígenes con hub m2: 57,046,115 / 57,573,489


## 5. Asignar hubs a destinos

In [10]:
dest_uniq = (
    df[['LatDestination', 'LngDestination']]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f'Destinos únicos con coords: {len(dest_uniq):,}')

Destinos únicos con coords: 575,767


In [11]:
dest_results = dest_uniq.apply(
    lambda r: assign_hub_m2(r['LatDestination'], r['LngDestination'], hubs_table),
    axis=1,
    result_type='expand'
)
dest_results.columns = [c.replace('_m2', '_dest_m2') for c in dest_results.columns]
dest_uniq = pd.concat([dest_uniq, dest_results], axis=1)

In [12]:
assert dest_uniq.duplicated(subset=['LatDestination', 'LngDestination']).sum() == 0, \
    "dest_uniq tiene duplicados en (LatDestination, LngDestination)!"

df = df.merge(
    dest_uniq,
    on=['LatDestination', 'LngDestination'],
    how='left'
)

assert len(df) == n_original, f"Row count cambió tras merge destinos: {len(df):,} != {n_original:,}"
print(f'✓ Filas después del merge destinos: {len(df):,} (igual a original)')
print(f'  Destinos con hub m2: {df["near_hub_dest_m2"].notna().sum():,} / {n_original:,}')

✓ Filas después del merge destinos: 57,573,489 (igual a original)
  Destinos con hub m2: 57,038,026 / 57,573,489


## 6. Verificación

In [13]:
sin_hub_con_coords = df[(df['near_hub_m2'].isna()) & (df['LatOrigin'].notna()) & (df['LngOrigin'].notna())]
print(f'Orígenes con coords pero sin hub m2: {len(sin_hub_con_coords):,}')

sin_hub_dest = df[(df['near_hub_dest_m2'].isna()) & (df['LatDestination'].notna()) & (df['LngDestination'].notna())]
print(f'Destinos con coords pero sin hub m2: {len(sin_hub_dest):,}')

Orígenes con coords pero sin hub m2: 0
Destinos con coords pero sin hub m2: 0


In [14]:
# Resumen de cobertura por tier
has_hub = df['near_hub_m2'].notna()
print(f'Cargas con hub origen m2: {has_hub.sum():,} ({100*has_hub.mean():.1f}%)')

if has_hub.sum() > 0:
    tier_map = hubs_table.set_index('hub_id')['tier'].to_dict()
    df['tier_hub_m2'] = df['near_hub_m2'].map(tier_map)
    print(df['tier_hub_m2'].value_counts())

Cargas con hub origen m2: 57,046,115 (99.1%)
tier_hub_m2
primaria      22157919
secundaria    19690297
terciario     15197899
Name: count, dtype: int64


## 7. Guardar resultado

In [15]:
# Guardar columnas clave
cols_to_keep = [
    'ID', 'HubOrigin', 'HubDestination',
    'LatOrigin', 'LngOrigin', 'LatDestination', 'LngDestination',
    'StateOrigin', 'StateDestination',
    'RatePerMile', 'Distance', 'Equip', 'Posted', 'Weight', 'Size',
    'near_hub_m2', 'near_hub_name_m2', 'tier_hub_m2',
    'hubs_50m_m2', 'hubs_100m_m2', 'hubs_150m_m2',
    'near_hub_dest_m2', 'near_hub_name_dest_m2',
    'hubs_50m_dest_m2', 'hubs_100m_dest_m2', 'hubs_150m_dest_m2'
]
cols_to_keep = [c for c in cols_to_keep if c in df.columns]
df[cols_to_keep].to_parquet('hubs_assign_m2.parquet', index=False)
print('Guardado: hubs_assign_m2.parquet')
df[cols_to_keep].head()

Guardado: hubs_assign_m2.parquet


,ID,HubOrigin,HubDestination,LatOrigin,LngOrigin,LatDestination,LngDestination,StateOrigin,StateDestination,RatePerMile,...,near_hub_name_m2,tier_hub_m2,hubs_50m_m2,hubs_100m_m2,hubs_150m_m2,near_hub_dest_m2,near_hub_name_dest_m2,hubs_50m_dest_m2,hubs_100m_dest_m2,hubs_150m_dest_m2
0,0.0,Grants Pass (OR),Stockton (CA),43.1802,-124.1841,37.5265,-121.9852,OR,CA,NaN,...,Cluster_2,terciario,[],[C2],[C2],86.0,Pico CA,[86.0],"[86.0, C16]","[86.0, C16, C18]"
1,1.0,Los Angeles (CA),Salem (OR),34.5277,-117.3536,45.1473,-122.8603,CA,OR,NaN,...,Pico CA,primaria,[],"[11.0, C10]","[11.0, C10, C4]",136.0,Pico OR,[136.0],"[136.0, C9, C8, C2]","[136.0, C9, C8, C2]"
2,2.0,Stockton (CA),Salem (OR),37.9765,-121.3109,44.0562,-121.3087,CA,OR,NaN,...,Cluster_16,terciario,[C16],"[86.0, C16]","[86.0, 64.0, C16, C18]",C9,Cluster_9,[C9],"[C9, C2]","[136.0, C9, C2]"
3,3.0,Stockton (CA),Yakima (WA),39.0817,-121.5549,45.8323,-119.2858,CA,OR,NaN,...,Cluster_16,terciario,[],"[C16, C18]","[86.0, 64.0, C16, C18]",C8,Cluster_8,[C8],[C8],"[C6, C9, C8]"
4,4.0,San Diego (CA),Salem (OR),33.9244,-117.2045,44.6274,-123.0966,CA,OR,NaN,...,Pico CA,primaria,[],"[11.0, C10, C4]","[11.0, C14, C10, C4]",C2,Cluster_2,[C2],"[136.0, C9, C2]","[136.0, C9, C8, C2]"
